# Milestone 3 – ResNet-18 on SVHN (No Code Smells)

ResNet-18 style model built in TensorFlow/Keras, fine-tuned on **SVHN**.
- EarlyStopping patience=2
- 30 epochs max
- 10 runs
- CodeCarbon tracking

In [ ]:
!pip install -q codecarbon

In [ ]:
# Fix codecarbon missing nordic_emissions.json
import codecarbon, os, json

_data_dir = os.path.join(os.path.dirname(codecarbon.__file__), 'data', 'private_infra')
os.makedirs(_data_dir, exist_ok=True)
_nordic_path = os.path.join(_data_dir, 'nordic_emissions.json')
if not os.path.exists(_nordic_path):
    with open(_nordic_path, 'w') as _f:
        json.dump({}, _f)
print('CodeCarbon data file ready:', _nordic_path)

In [ ]:
import numpy as np
import pandas as pd
import gc, os, warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation, Add,
    MaxPooling2D, GlobalAveragePooling2D, Dense
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

from sklearn.metrics import precision_score, recall_score
from codecarbon import EmissionsTracker

In [ ]:
from torchvision.datasets import SVHN

print('Downloading / loading SVHN dataset via torchvision...')
train_ds = SVHN(root='./data', split='train', download=True)
test_ds  = SVHN(root='./data', split='test',  download=True)

# .data -> (N, C, H, W) uint8; transpose to (N, H, W, C) for TF
X_train_np = train_ds.data.astype('float32').transpose(0,2,3,1) / 255.0
y_train_np = train_ds.labels.astype('int64')
X_test_np  = test_ds.data.astype('float32').transpose(0,2,3,1)  / 255.0
y_test_np  = test_ds.labels.astype('int64')

y_cat_train = to_categorical(y_train_np, 10)
y_cat_test  = to_categorical(y_test_np,  10)

print(f'X_train: {X_train_np.shape}  X_test: {X_test_np.shape}')

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x
    x = Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, strides=stride, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x

def build_resnet18(input_shape=(32, 32, 3), num_classes=10):
    METRICS = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
    inputs = Input(shape=input_shape)
    x = Conv2D(64, 3, strides=1, padding='same', use_bias=False)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    for filters, stride in [(64,1),(64,1),(128,2),(128,1),(256,2),(256,1),(512,2),(512,1)]:
        x = residual_block(x, filters, stride)
    x = GlobalAveragePooling2D()(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, outputs)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=METRICS)
    return model

## Training – 10 Runs

In [ ]:
os.makedirs('emissions', exist_ok=True)
results = []

for run in range(1, 11):
    print(f"\n{'='*60}")
    print(f'  Run {run}/10')
    print(f"{'='*60}")

    # Clean practice: reset graph between runs
    K.clear_session()

    tracker = EmissionsTracker(
        project_name=f'ResNet18_SVHN_Clean_Run{run}',
        output_dir='./emissions',
        save_to_file=True,
        log_level='warning'
    )
    tracker.start()

    model      = build_resnet18()
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

    history = model.fit(
        X_train_np, y_cat_train,
        epochs=30, batch_size=32,
        validation_data=(X_test_np, y_cat_test),
        callbacks=[early_stop],
        verbose=1
    )

    co2_kg     = tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed

    epoch_stopped = len(history.history['loss'])
    eval_res      = model.evaluate(X_test_np, y_cat_test, verbose=0)

    run_result = dict(
        run=run,
        epoch_stopped=epoch_stopped,
        test_loss=eval_res[0],
        test_accuracy=eval_res[1],
        test_precision=eval_res[2],
        test_recall=eval_res[3],
        energy_kwh=energy_kwh,
        co2_kg=co2_kg
    )
    results.append(run_result)
    print(f'Run {run} finished | Epoch stopped: {epoch_stopped} | '
          f'Acc: {eval_res[1]:.4f} | Energy: {energy_kwh:.6f} kWh | CO2: {co2_kg:.6f} kg')

    # Clean practice: free memory explicitly
    del model
    gc.collect()

## Results Summary

In [ ]:
print('\n' + '='*80)
print('FINAL RESULTS SUMMARY')
print('='*80)
header = f"{'Run':<5} {'EpochStopped':<14} {'Accuracy':<12} {'Precision':<12} {'Recall':<10} {'Energy(kWh)':<14} {'CO2(kg)':<12}"
print(header)
print('-'*80)
for r in results:
    print(f"{r['run']:<5} {r['epoch_stopped']:<14} {r['test_accuracy']:<12.4f} "
          f"{r['test_precision']:<12.4f} {r['test_recall']:<10.4f} "
          f"{r['energy_kwh']:<14.6f} {r['co2_kg']:<12.6f}")
print('-'*80)
avg_epoch  = np.mean([r['epoch_stopped']  for r in results])
avg_acc    = np.mean([r['test_accuracy']  for r in results])
avg_prec   = np.mean([r['test_precision'] for r in results])
avg_rec    = np.mean([r['test_recall']    for r in results])
avg_energy = np.mean([r['energy_kwh']     for r in results])
avg_co2    = np.mean([r['co2_kg']         for r in results])
print(f"{'AVG':<5} {avg_epoch:<14.1f} {avg_acc:<12.4f} {avg_prec:<12.4f} "
      f"{avg_rec:<10.4f} {avg_energy:<14.6f} {avg_co2:<12.6f}")
print('='*80)